# 01 — Problem Framing & Ingestion — Pokémon base stats (Generations 1–6)

**Stages 0 and 1 of the Tier A pipeline** (`DOCS/STRUCTURE.md`). Stage 0 is a thinking step, so it is
written down here *before* any data is touched: the question, the stakeholder, the issue tree, the SCR
hypothesis and the ghost deck. Stage 1 then loads the file exactly as delivered, proves what it is, and
records what it carries *before* anyone cleans anything.

The data review (`DOCS/pokemon_data_assessment.md`) was written from the dataset description alone; it
says so itself and marks its unchecked claims **[verify]**. This notebook checks each of them against the
raw CSV. Several do not hold: the file has **800 rows and 13 columns, not 721 × 11**; `Generation` and
`Legendary` are present; and the `Legendary` flag turns out to be inconsistent (§1.6).

## 0 — Problem framing

### 0.1 The business question (one sentence)

> **When designing a new roster of creatures, how much should a creature's type constrain its stats —
> its overall power and its stat profile — and can a model trained on Generations 1–6 check new designs
> for "off-type" stats?**

The review's headline question ("which two stats best separate types, and how much of type is explained
by stats at all?") is kept inside this one: it is the evidence for the last clause. The question above adds
the reason anyone would ask it — a design decision.

### 0.2 Stakeholder and decision

| | |
|---|---|
| **Primary stakeholder** | Lead creature-balance designer on a new monster-collector title, using the main-series roster (Generations 1–6) as the reference corpus |
| **Decision this feeds** | (1) Should stat lines be authored from **type-specific templates**? (2) What **power-budget rules** should creatures follow? (3) Should the tools team build a **stats → type classifier** as an automatic design check? |
| **What "done" looks like** | (1) A per-type yes/no on stat templates, with the template where the answer is yes; (2) a written power-budget ladder backed by the corpus; (3) a go / no-go on the classifier, judged against a baseline on an honest validation split |
| **Analysis type** | Rungs 1–2 (descriptive / diagnostic, **Path A**) plus a predictive check (**Path B**, Stages 4–6) for decision 3. No causal claim: stats do not "cause" type, and the file has no outcome |
| **Modality & grain** | Cross-sectional tabular. **One row = one species *or* one alternate form** (to be verified — the review suspected this). The analysis grain is set in `02` |

### 0.3 Scope boundaries — explicitly **out**

- "Which Pokémon is best?" or "who wins a battle?" — needs moves, abilities, levels and an outcome variable; none is here.
- Competitive viability (usage tiers) — not in the file.
- Generations 7 onwards — not in the file; conclusions are about Generations 1–6.
- Type effectiveness (weaknesses, resistances) — needs the type chart, which is external.

### 0.4 MECE issue tree

```
How much should type constrain stats, and can a model check new designs for off-type stats?
│
├── A. The file        — what is one row, and can rows be treated as independent?
│     A1 schema & review re-derivation       A2 forms (Mega / alternate) sharing a number
│     A3 flags (Legendary)                   A4 evolution families (non-independence)
│
├── B. Power (Total)   — what sets how strong a creature is?
│     B1 tier rules (legendary, Mega)        B2 type       B3 generation (power creep)
│
├── C. Shape           — given its power, does type set how the points are spread?
│     C1 size effect vs trade-offs           C2 per-type signatures       C3 natural archetypes
│
└── D. Prediction      — can stats recover type well enough to serve as a design check?
      D1 baseline & honest validation        D2 best pair of stats        D3 which types are recognisable
```

Branches are mutually exclusive by *object* (the instrument / the magnitude of the stat line / the
proportions of the stat line / a model built on both) and together exhaust the question. B and C split
the stat line cleanly: `Total` is magnitude, the six **shares** (stat ÷ Total) are shape.

### 0.5 SCR — Stage 0 draft (a hypothesis, to be revised in `06`)

| | Draft |
|---|---|
| **Situation** | The 721 species of Generations 1–6 each have six base stats and one or two of 18 types. Designers and players treat types as having "typical" stats: Steel is bulky, Electric is fast. |
| **Complication** | The dataset's own premise is that type cannot be told from Attack and Defense. If type barely constrains stats, type-based templates are folklore and a stats → type design check would not work. Nobody has measured how much it does. |
| **Resolution (hypothesis)** | Type explains little of a creature's *power* but some of its *shape*; a few types (Steel, Rock, Ghost, Dragon per the review) have clear signatures; a classifier beats the majority baseline but not by enough to act as a check. |

### 0.6 Ghost deck

```
SLIDE 1 — Governing Thought   [TBD after analysis]

PART I · The file
  "800 rows are [x] species plus [y] forms — and rows are [not] independent"
  [Placeholder: composition; review claims vs file]

PART II · Power
  "Total is set by [tier rules], not by type or generation"
  [Placeholder: Total by tier; Mega uplift; Total by generation]

PART III · Shape
  "[n] of 18 types carry a stat signature; the rest look like the average creature"
  [Placeholder: type × stat-share map (key exhibit candidate); variance explained]

PART IV · Prediction
  "Stats identify type [x]% of the time vs [y]% for the majority guess — [go / no-go] on a design check"
  [Placeholder: accuracy by validation protocol; best stat pair; per-type recognisability]

RECOMMENDATION — budget rules · templates · classifier go/no-go · data asks
APPENDIX — cleaning log, tests, model card, limits
```

### Stage 0 — Gate Checklist

- [x] Business question is a single, unambiguous sentence
- [x] Issue tree has ≥ 2 levels of decomposition and is MECE
- [x] "Done" is defined as concrete outcomes (per-type template yes/no, a budget ladder, a classifier go/no-go)
- [x] Primary stakeholder named, decision context documented
- [x] Scope boundaries explicit (battle outcomes, usage tiers, Gen 7+, type chart — all out)
- [x] SCR drafted · [x] Ghost deck exists with action-title placeholders · [x] Governing Thought drafted (tentative)

## 1 — Ingestion

### 1.1 Setup

In [1]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd()
while not (ROOT / "archive" / "Pokemon.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ARCHIVE = ROOT / "archive"
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
REPORTS, FIGS, MODELS = ROOT / "reports", ROOT / "reports" / "figures", ROOT / "models"
for p in (RAW, PROC, REPORTS, FIGS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

KF_PATH = REPORTS / "_key_figures.json"
STATS = ["hp", "atk", "def", "spa", "spd", "spe"]
STAT_LABEL = {"hp": "HP", "atk": "Attack", "def": "Defense", "spa": "Sp. Atk", "spd": "Sp. Def", "spe": "Speed"}
SHARES = [f"sh_{s}" for s in STATS]


def _plain(v):
    if isinstance(v, dict):
        return {str(k): _plain(x) for k, x in v.items()}
    if hasattr(v, "item"):
        return v.item()
    if isinstance(v, (list, tuple)):
        return [_plain(x) for x in v]
    return v


def kf_update(**kw):
    """Append to reports/_key_figures.json — the only place the report reads numbers from."""
    kf = json.loads(KF_PATH.read_text(encoding="utf-8")) if KF_PATH.exists() else {}
    kf.update({k: _plain(v) for k, v in kw.items()})
    KF_PATH.write_text(json.dumps(kf, indent=1, sort_keys=True, ensure_ascii=False), encoding="utf-8")
    print(f"  _key_figures.json <- {', '.join(kw)}  ({len(kf)} figures total)")

import hashlib, re, shutil
from datetime import datetime, timezone

SRC = ARCHIVE / "Pokemon.csv"
if KF_PATH.exists():
    KF_PATH.unlink()          # 01 starts the key-figures file afresh; 02–05 append to it
print(f"project root: {ROOT.name}")
print(f"source file : {SRC.relative_to(ROOT)}  ({SRC.stat().st_size:,} bytes)")

project root: Pokemon with stats
source file : archive\Pokemon.csv  (44,028 bytes)


### 1.2 Load exactly as delivered, preserve an immutable copy, log lineage

No type coercion, no renaming, no cleaning. The SHA-256 is recorded so any later run can prove it is
reading the same bytes.

In [2]:
raw = pd.read_csv(SRC, encoding="utf-8")
sha = hashlib.sha256(SRC.read_bytes()).hexdigest()

immutable = RAW / "pokemon_raw_immutable.csv"
shutil.copyfile(SRC, immutable)
assert hashlib.sha256(immutable.read_bytes()).hexdigest() == sha

lineage = pd.DataFrame([{
    "file": SRC.name, "source": "Kaggle 'Pokemon with stats' (abcsds); compiled from pokemondb.net, "
                                "pokemon.com and bulbapedia per the dataset page",
    "pulled_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "rows": raw.shape[0], "cols": raw.shape[1], "bytes": SRC.stat().st_size, "sha256": sha,
}])
lineage.to_csv(RAW / "_lineage.csv", index=False)
print(lineage.T.to_string(header=False))
print()
print(raw.dtypes.to_string())
raw.head(6)

file                                                                                                               Pokemon.csv
source      Kaggle 'Pokemon with stats' (abcsds); compiled from pokemondb.net, pokemon.com and bulbapedia per the dataset page
pulled_utc                                                                                                    2026-10-14 11:38
rows                                                                                                                       800
cols                                                                                                                        13
bytes                                                                                                                    44028
sha256                                                        02eabc49d3c70bc1e150891bdb9acb254910adfdfc12a6020c7b7117b2eca32f

#             int64
Name            str
Type 1          str
Type 2          str
Total         int64
HP        

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False
5,5,Charmeleon,Fire,NaN,405,58,64,58,80,65,80,1,False


### 1.3 Schema contract

A written contract of what each column must be. Every check is run; failures are recorded, not silently
fixed.

In [3]:
SIX = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
REVIEW_COLS = ["#", "Name", "Type 1", "Type 2", "Total"] + SIX
TYPES = {"Normal", "Fire", "Water", "Grass", "Electric", "Ice", "Fighting", "Poison", "Ground", "Flying",
         "Psychic", "Bug", "Rock", "Ghost", "Dragon", "Dark", "Steel", "Fairy"}
GEN_RANGES = {1: (1, 151), 2: (152, 251), 3: (252, 386), 4: (387, 493), 5: (494, 649), 6: (650, 721)}

gen_from_dex = pd.cut(raw["#"], [0, 151, 251, 386, 493, 649, 721], labels=[1, 2, 3, 4, 5, 6]).astype(int)
t2 = raw["Type 2"].dropna()

contract = [
    ("columns as described in the review (11)", list(raw.columns) == REVIEW_COLS,
     f"{raw.shape[1]} columns: also {[c for c in raw.columns if c not in REVIEW_COLS]}"),
    ("no missing values outside Type 2", raw.drop(columns="Type 2").notna().all().all(),
     f"{int(raw.drop(columns='Type 2').isna().sum().sum())} nulls"),
    ("'#' is a unique key", raw["#"].is_unique, f"{raw['#'].nunique()} distinct values in {len(raw)} rows"),
    ("'#' contiguous 1..721", set(raw["#"]) == set(range(1, 722)), f"{raw['#'].min()}–{raw['#'].max()}"),
    ("Name unique", raw["Name"].is_unique, f"{raw['Name'].nunique()} distinct"),
    ("Type 1 / Type 2 in the 18 canonical types", set(raw["Type 1"]) <= TYPES and set(t2) <= TYPES,
     f"{raw['Type 1'].nunique()} / {t2.nunique()} types used"),
    ("Type 2 never repeats Type 1", (raw["Type 1"] != raw["Type 2"]).all(), "checked row by row"),
    ("Total == sum of the six stats", (raw[SIX].sum(axis=1) == raw["Total"]).all(),
     f"{int((raw[SIX].sum(axis=1) != raw['Total']).sum())} mismatches"),
    ("every stat an integer in 1–255", raw[SIX].ge(1).all().all() and raw[SIX].le(255).all().all(),
     f"{raw[SIX].min().min()}–{raw[SIX].max().max()}"),
    ("Generation matches the '#' ranges", (gen_from_dex == raw["Generation"]).all(),
     f"{int((gen_from_dex != raw['Generation']).sum())} mismatches"),
    ("Legendary is boolean", raw["Legendary"].dtype == bool, str(raw["Legendary"].dtype)),
]
schema = pd.DataFrame(contract, columns=["check", "pass", "detail"])
schema.to_csv(RAW / "_schema_validation.csv", index=False)
print(f"{schema['pass'].sum()} of {len(schema)} contract checks pass\n")
schema

9 of 11 contract checks pass



,check,pass,detail
0,columns as described in the review (11),False,"13 columns: also ['Generation', 'Legendary']"
1,no missing values outside Type 2,True,0 nulls
2,'#' is a unique key,False,721 distinct values in 800 rows
3,'#' contiguous 1..721,True,1–721
4,Name unique,True,800 distinct
5,Type 1 / Type 2 in the 18 canonical types,True,18 / 18 types used
6,Type 2 never repeats Type 1,True,checked row by row
7,Total == sum of the six stats,True,0 mismatches
8,every stat an integer in 1–255,True,1–255
9,Generation matches the '#' ranges,True,0 mismatches


Two checks fail, and both are the review's **[verify]** items. The file has 13 columns (it carries
`Generation` and `Legendary`, which the review expected to be missing), and `#` repeats, so it is not a
unique key. Everything else holds: no stray nulls, canonical types, `Total` always equals the sum of the
six stats, and `Generation` agrees with the `#` ranges exactly.

### 1.4 Re-deriving the data review's claims

In [4]:
first = ~raw["#"].duplicated()
sp = raw[first]
water_800 = (raw["Type 1"] == "Water").mean()
water_721 = (sp["Type 1"] == "Water").mean()

checks = pd.DataFrame([
    ("rows × columns", "~721 × 11", f"{raw.shape[0]} × {raw.shape[1]}"),
    ("'#' repeats for forms", "possible [verify]", f"yes: {int(raw['#'].duplicated().sum())} extra rows share a '#'"),
    ("Generation / Legendary present", "absent [verify]", "both present"),
    ("Type 2 blank", "roughly half", f"{raw['Type 2'].isna().mean():.1%} of rows; {sp['Type 2'].isna().mean():.1%} of species"),
    ("Total redundant with the six stats", "yes", "yes: 0 mismatches"),
    ("Type 1 largest classes", "Water, Normal, Grass",
     ", ".join(f"{k} {v}" for k, v in raw["Type 1"].value_counts().head(3).items())),
    ("Flying as Type 1", "very rare", f"{(raw['Type 1'] == 'Flying').sum()} rows"),
    ("majority-class accuracy (Water)", "roughly 13–14%", f"{water_800:.1%} of rows; {water_721:.1%} of species"),
    ("stats right-skewed", "yes", " / ".join(f"{raw[c].skew():+.2f}" for c in SIX) + "  (HP…Speed)"),
    ("stats positively correlated", "yes",
     f"{int((raw[SIX].corr().where(~np.eye(6, dtype=bool)).stack() > 0).sum() // 2)} of 15 pairs > 0"),
    ("generation derivable from '#'", "1–151, 152–251, …", "exact match on all 800 rows"),
], columns=["claim", "review says", "file says"])
checks

,claim,review says,file says
0,rows × columns,~721 × 11,800 × 13
1,'#' repeats for forms,possible [verify],yes: 79 extra rows share a '#'
2,Generation / Legendary present,absent [verify],both present
3,Type 2 blank,roughly half,48.2% of rows; 51.5% of species
4,Total redundant with the six stats,yes,yes: 0 mismatches
5,Type 1 largest classes,"Water, Normal, Grass","Water 112, Normal 98, Grass 70"
6,Flying as Type 1,very rare,4 rows
7,majority-class accuracy (Water),roughly 13–14%,14.0% of rows; 14.6% of species
8,stats right-skewed,yes,+1.57 / +0.55 / +1.16 / +0.74 / +0.85 / +0.36 ...
9,stats positively correlated,yes,15 of 15 pairs > 0


The review's statements about *behaviour* hold (skew, positive correlation, class imbalance, Water as
majority at ~14%). Its statements about *structure* do not: 800 rows, 13 columns, `#` repeats. The next two
sections establish what the extra 79 rows are and whether the two extra columns can be trusted.

### 1.5 What one row is — species, Mega Evolutions and alternate forms

In [5]:
dup = raw[raw["#"].duplicated(keep=False)]
joined = raw[raw["Name"].str.contains(r"[a-z][A-Z]", regex=True)]
print(f"rows whose '#' repeats: {len(dup)} rows across {dup['#'].nunique()} species numbers")
print(f"names with a glued-on form label (lower-case letter followed by capital): {len(joined)}")
print("  e.g.", ", ".join(joined["Name"].head(8)))

is_mega = raw["Name"].str.contains("Mega ", regex=False)
kind = np.where(first, "species (first row of its '#')", np.where(is_mega, "Mega Evolution", "alternate form"))
comp = pd.Series(kind).value_counts()
print("\n" + comp.to_string())

base_total = sp.set_index("#")["Total"]
mega = raw[is_mega].copy()
mega["uplift"] = mega["Total"].to_numpy() - base_total.loc[mega["#"]].to_numpy()
mega["type1_changes"] = mega["Type 1"].to_numpy() != sp.set_index("#").loc[mega["#"], "Type 1"].to_numpy()
print(f"\nMega uplift in Total: {mega['uplift'].value_counts().to_dict()}")
print(f"Megas whose Type 1 differs from their species: {int(mega['type1_changes'].sum())}")

rows whose '#' repeats: 144 rows across 65 species numbers
names with a glued-on form label (lower-case letter followed by capital): 94
  e.g. VenusaurMega Venusaur, CharizardMega Charizard X, CharizardMega Charizard Y, BlastoiseMega Blastoise, BeedrillMega Beedrill, PidgeotMega Pidgeot, AlakazamMega Alakazam, SlowbroMega Slowbro

species (first row of its '#')    721
Mega Evolution                     48
alternate form                     31

Mega uplift in Total: {100: 47, 90: 1}
Megas whose Type 1 differs from their species: 0


The 800 rows are **721 species** (the first row of each `#`), **48 Mega Evolutions** and **31 alternate
forms** (Primal Kyogre/Groudon, Deoxys and Rotom formes, Pumpkaboo sizes, …). The form label is glued onto
the species name without a separator (`VenusaurMega Venusaur`), which `02` splits.

Every Mega keeps its species' `Type 1`, and **47 of 48 add exactly 100 to `Total`**. So a form is a
near-copy of its species: same number, same primary type, a fixed stat bump. Left in, forms would let
a model "recognise" a species it has already seen. **Analysis grain: the 721 species.** Forms are kept,
flagged, and used only where they are the subject (the Mega rule in Part II).

### 1.6 The `Legendary` flag is inconsistent

In [6]:
leg = sp[sp["Legendary"]]
print(f"species flagged Legendary: {len(leg)}; lowest Total among them: {leg['Total'].min()}")
hi = sp[(sp["Total"] >= 580) & ~sp["Legendary"]][["#", "Name", "Total", "Generation"]]
print(f"\nspecies NOT flagged Legendary with Total >= 580: {len(hi)}")
hi

species flagged Legendary: 46; lowest Total among them: 580

species NOT flagged Legendary with Total >= 580: 15


,#,Name,Total,Generation
161,149,Dragonite,600,1
165,151,Mew,600,1
267,248,Tyranitar,600,2
271,251,Celebi,600,2
313,289,Slaking,670,3
408,373,Salamence,600,3
412,376,Metagross,600,3
493,445,Garchomp,600,4
546,488,Cresselia,600,4
548,490,Manaphy,600,4


Every flagged legendary species has `Total` ≥ 580. Fifteen species reach that level without the flag. Seven
are the ordinary "pseudo-legendaries" at exactly 600 (Dragonite, Tyranitar, Salamence, Metagross, Garchomp,
Hydreigon, Goodra), and one is Slaking (670). The other seven, **Mew, Celebi, Cresselia, Manaphy, Keldeo,
Meloetta and Genesect**, are classed by the games themselves as legendary or *mythical*. Meanwhile other
mythicals (Jirachi, Darkrai, Arceus, Victini, Diancie, Hoopa, Volcanion) **are** flagged. The flag
therefore mixes two definitions.

This is a domain correction, so it is written down as an explicit list rather than inferred:

In [7]:
# The games' own classification (legendary or mythical) for species this file flags False.
# Phione (Total 480) is mythical too, so the fix is not "Total >= 580".
CANON_MISFLAGGED = ["Mew", "Celebi", "Cresselia", "Phione", "Manaphy", "Keldeo", "Meloetta", "Genesect"]
mis = sp[sp["Name"].str.match(r"^(%s)(?![a-z])" % "|".join(CANON_MISFLAGGED))][["#", "Name", "Total", "Legendary"]]
assert len(mis) == len(CANON_MISFLAGGED) and not mis["Legendary"].any()
mythical_flagged = ["Jirachi", "Deoxys", "Darkrai", "Shaymin", "Arceus", "Victini", "Diancie", "Hoopa", "Volcanion"]
mf = sp[sp["Name"].str.match(r"^(%s)(?![a-z])" % "|".join(mythical_flagged))]
print(f"mythicals flagged True : {mf['Legendary'].sum()} of {len(mf)}  ({', '.join(mf['Name'].str.extract(r'^([A-Z][a-z]+)')[0])})")
print(f"canon legendary/mythical flagged False: {len(mis)}")
mis

mythicals flagged True : 9 of 9  (Jirachi, Deoxys, Darkrai, Shaymin, Arceus, Victini, Diancie, Hoopa, Volcanion)
canon legendary/mythical flagged False: 8


,#,Name,Total,Legendary
165,151,Mew,600,False
271,251,Celebi,600,False
546,488,Cresselia,600,False
547,489,Phione,480,False
548,490,Manaphy,600,False
713,647,KeldeoOrdinary Forme,580,False
715,648,MeloettaAria Forme,600,False
717,649,Genesect,600,False


`02` builds `legendary_canon` = `Legendary` **or** one of those eight. That gives 54 legendary or mythical
species. The raw flag is kept beside it, so every result can be checked against either definition.

### 1.7 Evolution families make rows non-independent

No column records evolution, but the Pokédex numbers evolution lines consecutively (Bulbasaur 1, Ivysaur 2,
Venusaur 3), and a line usually keeps its type and scales its stats up roughly in proportion. If so, a
species' nearest neighbour in stat *shape* should often be a dex neighbour.

In [8]:
from sklearn.neighbors import NearestNeighbors

shape = sp[SIX].to_numpy(float)
shape = shape / shape.sum(axis=1, keepdims=True)
nn = NearestNeighbors(n_neighbors=2).fit(shape)
nb2 = nn.kneighbors(shape, return_distance=False)
self_first = nb2[:, 0] == np.arange(len(shape))       # exact shape ties can return self second
idx = np.where(self_first, nb2[:, 1], nb2[:, 0])
dex = sp["#"].to_numpy()
near_rate = float((np.abs(dex[idx] - dex) <= 2).mean())

rng = np.random.default_rng(RANDOM_STATE)
null = np.array([(np.abs(p[idx] - p) <= 2).mean() for p in (rng.permutation(dex) for _ in range(2000))])
print(f"nearest stat-shape neighbour within ±2 dex numbers: {near_rate:.1%}")
print(f"same, with dex numbers shuffled (2,000 runs): mean {null.mean():.2%}, max {null.max():.2%}")

nearest stat-shape neighbour within ±2 dex numbers: 26.1%
same, with dex numbers shuffled (2,000 runs): mean 0.56%, max 1.80%


A species' closest match in stat proportions sits within two Pokédex numbers of it **26% of the
time, against about 0.6% by chance**. These are evolution families. They share a type and a stat shape, so
a random train/test split puts siblings on both sides and lets a model recognise the family instead of the
type. The review recommends a *stratified* split. That is not enough here: `05` must split **by family**.
`02` builds a family proxy from the dex order.

### 1.8 Governance

In [9]:
gov = pd.DataFrame([
    ("PII present", "none — fictional creatures"),
    ("intellectual property", "stats belong to Nintendo / Game Freak / The Pokémon Company; fan-compiled public dataset, portfolio use"),
    ("lineage", "merged by the uploader from three fan sites; no per-row source, no conflict handling recorded"),
    ("labels", "Legendary flag mixes two definitions (§1.6); corrected column added in 02, raw kept"),
    ("version", "Generation 6 era values; later games changed some base stats — conclusions are about Gens 1–6 as recorded here"),
    ("fairness audit", "not applicable — no people affected"),
], columns=["item", "assessment"])
gov.to_csv(RAW / "_governance.csv", index=False)
gov

,item,assessment
0,PII present,none — fictional creatures
1,intellectual property,stats belong to Nintendo / Game Freak / The Po...
2,lineage,merged by the uploader from three fan sites; n...
3,labels,Legendary flag mixes two definitions (§1.6); c...
4,version,Generation 6 era values; later games changed s...
5,fairness audit,not applicable — no people affected


In [10]:
kf_update(
    n_rows_raw=len(raw), n_cols_raw=raw.shape[1], sha256=sha,
    n_species=int(first.sum()), n_mega=int(is_mega.sum()), n_alt=int((~first & ~is_mega).sum()),
    n_dex_repeats=int(raw["#"].duplicated().sum()),
    n_schema_checks=len(schema), n_schema_pass=int(schema["pass"].sum()),
    type2_null_rows=float(raw["Type 2"].isna().mean()), single_type_species=float(sp["Type 2"].isna().mean()),
    water_share_rows=float(water_800), water_share_species=float(water_721),
    n_flying_type1=int((raw["Type 1"] == "Flying").sum()),
    mega_plus100=int((mega["uplift"] == 100).sum()), mega_other_uplift=int(mega.loc[mega["uplift"] != 100, "uplift"].iloc[0]),
    mega_other_name=str(mega.loc[mega["uplift"] != 100, "Name"].iloc[0]),
    n_legendary_raw=int(sp["Legendary"].sum()), n_canon_misflagged=len(CANON_MISFLAGGED),
    n_mythical_flagged=int(mf["Legendary"].sum()),
    n_hi_total_unflagged=len(hi),
    nn_dex_near_rate=near_rate, nn_dex_near_null_mean=float(null.mean()), nn_dex_near_null_max=float(null.max()),
)

  _key_figures.json <- n_rows_raw, n_cols_raw, sha256, n_species, n_mega, n_alt, n_dex_repeats, n_schema_checks, n_schema_pass, type2_null_rows, single_type_species, water_share_rows, water_share_species, n_flying_type1, mega_plus100, mega_other_uplift, mega_other_name, n_legendary_raw, n_canon_misflagged, n_mythical_flagged, n_hi_total_unflagged, nn_dex_near_rate, nn_dex_near_null_mean, nn_dex_near_null_max  (24 figures total)


## Stage 1 — Gate Checklist

- [x] Raw data saved to `data/raw/pokemon_raw_immutable.csv`, hash-verified
- [x] Source, timestamp, row/column counts and SHA-256 logged (`data/raw/_lineage.csv`)
- [x] Schema checked against a written contract; the two failures (13 columns, `#` not unique) documented, not fixed
- [x] Governance review recorded (`data/raw/_governance.csv`): no PII; IP and lineage caveats stated
- [x] Data review re-derived from the raw file: behavioural claims reproduce; **structural claims do not**
      (800 × 13, `#` repeats, Generation and Legendary present), and the Legendary flag is inconsistent

**Carried into `02`:** split glued form names · analysis grain = 721 species, forms flagged · build
`legendary_canon` · build an evolution-family proxy for grouped validation · derived columns from review §7.